In [2]:
DROP_CONFIG = {
    "cuti": [
        # "roster",
        # "nama_kolom_lain"
    ],
    "tanggal": [
        # "roster",
        # "nama_kolom_lain"
    ]
}



import pandas as pd
from pathlib import Path
import re

DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed")
REPORT_DIR = Path("output/reports")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "cuti": DATA_DIR / "Cuti.xlsx",
    "tanggal": DATA_DIR / "Tanggal.xlsx"
}

# Edit separately after reviewing profiling results.
DROP_CONFIG = {
    "cuti": [
        "roster",
        "unnamed_18",
        "unnamed_19",
        "unnamed_20",
        "attendance_allowance_1",
        "attendance_penalty_1",
        "shift_allowance_1",
        "organization_group_code",
        "organization_group_name"
    ],
    "tanggal": [
        "roster",
        "spl_day_type",
        "difference_in_work_schedule_with_actual_attendance",
        "difference_in_check_out_time",
        "check_in_attendance_terminal",
        "check_out_attendance_terminal",
        "attendance_location",
        "failed_attendance_posting",
        "description",
        "holiday_type",
        "check_in_attendance_address_mob",
        "check_out_attendance_address_mob",
        "position_code",
        "position_name"
    ]
}
# Do not drop these columns without explicit review.
PROTECTED_COLUMNS = {
    "employee_id",
    "attendance_date"
}


def normalize_column(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def clean_dataset(df):
    df = df.copy()

    # Normalize column names
    original_columns = df.columns.tolist()
    normalized_columns = [
        normalize_column(col) for col in original_columns
    ]

    # Stop if normalization creates duplicate names.
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            "Duplicate column names after normalization. "
            "Review the original column names."
        )

    df.columns = normalized_columns

    # Clean text without changing missing values
    for col in df.select_dtypes(include=["object", "string"]):
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace("", pd.NA)

    # Employee ID should remain text
    if "employee_id" in df.columns:
        df["employee_id"] = (
            df["employee_id"].astype("string").str.strip()
        )

    # Attendance date should be datetime
    if "attendance_date" in df.columns:
        df["attendance_date"] = pd.to_datetime(
            df["attendance_date"],
            errors="coerce"
        )

    return df


def drop_selected_columns(df, source_name):
    df = df.copy()
    columns_to_drop = DROP_CONFIG[source_name]

    unknown = set(columns_to_drop) - set(df.columns)
    if unknown:
        raise ValueError(
            f"Unknown columns in {source_name}: {sorted(unknown)}"
        )

    protected = set(columns_to_drop) & PROTECTED_COLUMNS
    if protected:
        raise ValueError(
            f"Protected columns cannot be dropped: {protected}"
        )

    drop_log = []

    for col in columns_to_drop:
        drop_log.append({
            "source": source_name,
            "column": col,
            "action": "DROP",
            "missing_before": int(df[col].isna().sum()),
            "reason": "Manual selection"
        })

        df = df.drop(columns=[col])

    return df, drop_log


all_drop_logs = []
all_validation = []

for source_name, file_path in FILES.items():
    print(f"\nProcessing: {source_name}")

    df_raw = pd.read_excel(
        file_path,
        dtype={"Employee ID": "string"}
    )

    original_rows = len(df_raw)
    original_cols = len(df_raw.columns)

    # Cleaning first
    df_cleaned = clean_dataset(df_raw)

    # Dropping separately
    df_final, drop_log = drop_selected_columns(
        df_cleaned, source_name
    )

    all_drop_logs.extend(drop_log)

    # Validate output
    all_validation.append({
        "source": source_name,
        "rows_before": original_rows,
        "rows_after": len(df_final),
        "columns_before": original_cols,
        "columns_after": len(df_final.columns),
        "missing_employee_id": int(
            df_final["employee_id"].isna().sum()
        ),
        "missing_attendance_date": int(
            df_final["attendance_date"].isna().sum()
        ),
        "duplicate_employee_date": int(
            df_final.duplicated(
                ["employee_id", "attendance_date"]
            ).sum()
        )
    })

    output_path = (
        OUTPUT_DIR / f"{source_name}_cleaned.xlsx"
    )

    df_final.to_excel(output_path, index=False)

    print(f"Saved: {output_path}")
    print(f"Rows: {len(df_final)}")
    print(f"Columns: {len(df_final.columns)}")


pd.DataFrame(
    all_drop_logs,
    columns=[
        "source", "column", "action",
        "missing_before", "reason"
    ]
).to_excel(
    REPORT_DIR / "dropping_log.xlsx",
    index=False
)

pd.DataFrame(all_validation).to_excel(
    REPORT_DIR / "cleaning_validation.xlsx",
    index=False
)

print("\nCleaning completed. No datasets were merged.")



Processing: cuti
Saved: data\processed\cuti_cleaned.xlsx
Rows: 30295
Columns: 74

Processing: tanggal
Saved: data\processed\tanggal_cleaned.xlsx
Rows: 171605
Columns: 55

Cleaning completed. No datasets were merged.
